# WTI Crude Oil — Interactive Agent Session (Notebook 7 of 7)

> **Part 7 of 7.** Optional — run after Notebook 5 (training) and 6 (eval).

This notebook shows how to interact with the trained adaptive agent  
in real time — both via a quick in-notebook demo and via `adk web`,  
the ADK browser UI.

The agent you interact with here has the strategy it learned in Notebook 5.  
You can give it real prediction requests, send it resolutions, ask it to  
review its own strategy, or pose open questions about the market.

---
## 0. Setup & State Check

In [ ]:
import asyncio
import warnings
from pathlib import Path

import yaml

from aieng.forecasting.methods.agentic import build_adk_agent
from aieng.forecasting.methods.agentic.adk_runner import AdkTextRunner, AdkTextRunnerConfig
from energy_oil_forecasting.adaptive_agent import build_wti_adaptive_config

warnings.filterwarnings('ignore')

_SKILLS_ROOT = Path('adaptive_agent/skills')
STATS_STRATEGY_DIR = _SKILLS_ROOT / 'wti-strategy-stats'
NEWS_STRATEGY_DIR  = _SKILLS_ROOT / 'wti-strategy-news'

AGENT_MODEL = 'gemini-3.1-flash-preview'

# ── Which variant to use for interactive demo ─────────────────────────────────
# Change to NEWS_STRATEGY_DIR to demo the news-grounded variant.
DEMO_STRATEGY_DIR = STATS_STRATEGY_DIR

# ── Show current strategy state ───────────────────────────────────────────────
state = yaml.safe_load((DEMO_STRATEGY_DIR / 'skill_state.yaml').read_text())
print(f'Strategy: {DEMO_STRATEGY_DIR.name}')
print(f'  Observations:            {len(state.get("observations", []))}')
print(f'  Hypotheses:              {len(state.get("hypotheses", []))}')
print(f'  Calibration corrections: {len(state.get("calibration_corrections", []))}')
print(f'  Version entries:         {len(state.get("version_history", []))}')

---
## 1. The Four Message Types

The adaptive agent handles four kinds of messages. Each triggers a different
response pattern and may or may not involve mutation tool calls.

| Message type | When to send | What to expect |
|---|---|---|
| **Prediction request** | You need a WTI forecast | Structured forecast with point estimate + intervals |
| **Resolution** | An earlier forecast period has resolved | Agent reviews error; may call `record_observation` or `record_hypothesis_outcome` |
| **Self-review request** | After several resolutions | Agent reflects on patterns; may open a hypothesis or graduate one |
| **Open question** | Anything else | Contextual response using search + code exec |

### Example prompts to try

**Prediction request:**
```
Please give me a WTI crude oil price forecast as of today.
I need point estimates and 80% prediction intervals for 5, 10, and 21 business days ahead.
```

**Resolution:**
```
Following up on a forecast you made two weeks ago: the actual WTI price on
[DATE] was $[PRICE]. Your point forecast was $[FORECAST] with an 80% interval
of [$LOWER, $UPPER]. Please review this outcome.
```

**Self-review request:**
```
Please review the observations and hypotheses in your strategy. Are there any
hypotheses that now have enough evidence to graduate to a calibration correction?
```

**Open question:**
```
What do you think is the most important factor driving WTI prices right now,
and how does your current strategy account for it?
```

---
## 2. In-Notebook Demo

Build the agent, send one prediction request, and print the response.  
This is the same interface the training notebooks used — you can send any  
of the four message types from here.

In [ ]:
config = build_wti_adaptive_config(
    model=AGENT_MODEL,
    strategy_dir=DEMO_STRATEGY_DIR,
)
agent = build_adk_agent(config)
runner = AdkTextRunner(
    agent,
    config=AdkTextRunnerConfig(
        app_name='wti_interactive',
        fresh_session_per_message=False,  # sticky session for multi-turn
    ),
)
print(f'Agent ready: {DEMO_STRATEGY_DIR.name}')

In [ ]:
# ── Send a prediction request ────────────────────────────────────────────────
# Modify the prompt below and re-run this cell for any of the four message types.

_DEMO_PROMPT = (
    'Please give me a WTI crude oil price forecast as of today. '
    'I need point estimates and 80% prediction intervals for '
    '5, 10, and 21 business days ahead. Please also briefly '
    'note which calibration corrections from your strategy, if any, '
    'you applied to this forecast.'
)

print('Sending prediction request...\n')
reply = asyncio.run(runner.run_text_async(_DEMO_PROMPT))
print(reply)

---
## 3. Interactive Session via `adk web`

For a full conversational experience — with the ADK trace panel showing  
every tool call and the agent's reasoning — use `adk web`.

### Starting the web UI

From the repo root, in a terminal:

```bash
cd implementations/energy_oil_forecasting
uv run adk web adaptive_agent/
```

This starts a local server at `http://localhost:8000`. Open it in your browser.

The default agent uses `wti-strategy/` (the base variant). To use a trained variant,
you can temporarily copy the trained `skill_state.yaml` over the default:

```bash
# Use the stats-trained variant
cp adaptive_agent/skills/wti-strategy-stats/skill_state.yaml \
   adaptive_agent/skills/wti-strategy/skill_state.yaml
uv run python -c "
from pathlib import Path
from aieng.forecasting.methods.agentic.adaptive_skill import AdaptiveSkillStore
from energy_oil_forecasting.adaptive_agent.skill_state import WtiStrategyState
store = AdaptiveSkillStore(Path('adaptive_agent/skills/wti-strategy'), WtiStrategyState)
store.save(store.load())  # re-render SKILL.md
"
```

### What to watch for

- The **ADK trace panel** (right sidebar) shows every tool call: search queries,
  code execution, and skill mutation calls (`record_observation`, etc.).
- After a resolution message, check `adaptive_agent/skills/wti-strategy/SKILL.md`
  to see whether the agent updated its strategy.
- The `.history/` directory inside each strategy dir contains timestamped backups
  of every `skill_state.yaml` — a full audit trail.

---
## 4. Multi-Turn Example (In-Notebook)

The `runner` uses a sticky session (`fresh_session_per_message=False`),  
so you can have a multi-turn conversation. The cell below shows a  
predict → resolve sequence.

In [ ]:
# ── Multi-turn: predict then resolve ─────────────────────────────────────────
# Step 1: prediction request (replace dates/prices with real values)
_PREDICT_PROMPT = (
    'Please forecast WTI crude oil prices as of today for '
    '5, 10, and 21 business days ahead. Include point estimates '
    'and 80% prediction intervals.'
)

# Uncomment to run:
# print('Step 1 — Prediction request')
# reply1 = asyncio.run(runner.run_text_async(_PREDICT_PROMPT))
# print(reply1)

# Step 2: resolution (fill in actual price after the horizon resolves)
_RESOLUTION_PROMPT = (
    'Resolution: the actual WTI price on [DATE] was $[ACTUAL]. '
    'Your point forecast for that date was $[FORECAST] with an 80% '
    'interval of [$LOWER, $UPPER]. Please review this outcome and '
    'record any relevant observations.'
)

# Uncomment to run:
# print('\nStep 2 — Resolution')
# reply2 = asyncio.run(runner.run_text_async(_RESOLUTION_PROMPT))
# print(reply2)

print('Multi-turn cells ready. Uncomment and fill in dates/prices to run.')